In [ ]:
!git clone https://<TOKEN>@github.com/<username>/<repo>.git

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# Main — CSR × User Simulation 오케스트레이션

```
builder_mvp.ipynb  →  app, initial_state, config 제공 (while 루프 셀 제거 필요)
user_sim.ipynb     →  UserSimAgent, PERSONA_TEMPLATES
main.ipynb         →  Queue 기반으로 두 시스템 연결
```

**주의**: `builder_mvp.ipynb`의 마지막 셀 (while 루프)은 주석 처리하거나 삭제하세요.  
루프 로직은 이 노트북이 담당합니다.

## 1. 노트북 로드

In [ ]:
!ls -l

In [ ]:
# CSR 시스템 로드 (app, initial_state, config 가 네임스페이스로 올라옴)
%run ./pipeline/builder_mvp.ipynb

In [ ]:
# User Sim 로드 (UserSimAgent, PERSONA_TEMPLATES 가 네임스페이스로 올라옴)
%run ./simulation/user_sim_test.ipynb

## 2. 오케스트레이션 설정

In [ ]:
import queue
import threading
import json
import copy
from langgraph.types import Command

# 통신 큐
user_to_csr = queue.Queue()   # UserSim → CSR (사용자 응답)
csr_to_user = queue.Queue()   # CSR → UserSim (질문 or 완료 신호)

# 결과 수집
eval_results = []

print("큐 초기화 완료")

## 3. CSR 실행 함수

`input()` 대신 `user_to_csr` 큐에서 읽어옵니다.

In [ ]:
def run_csr(thread_id: str):
    """
    CSR LangGraph 앱을 실행하며 interrupt 발생 시
    input() 대신 큐에서 사용자 응답을 가져옵니다.
    """
    session_config = {"configurable": {"thread_id": thread_id}}
    state = copy.deepcopy(initial_state)
    result = app.invoke(state, config=session_config)

    while True:
        if "__interrupt__" in result:
            question = result["__interrupt__"][0].value
            csr_to_user.put(question)           # UserSim에 질문 전달
            user_input = user_to_csr.get()      # UserSim 응답 대기 (블로킹)
            result = app.invoke(Command(resume=user_input), config=session_config)
        else:
            csr_to_user.put({"__done__": True, "result": result})
            break

print("run_csr 정의 완료")

## 4. UserSim 실행 함수

In [ ]:
def run_user_sim(persona: dict, result_collector: list):
    """
    UserSimAgent를 실행하며 CSR 질문에 자동 응답합니다.
    세션이 끝나면 결과를 result_collector에 추가합니다.
    """
    agent = UserSimAgent(persona=persona, verbose=True)

    while True:
        message = csr_to_user.get()  # CSR 메시지 대기 (블로킹)

        if isinstance(message, dict) and message.get("__done__"):
            csr_result = message["result"]
            result_collector.append({
                "persona":         persona,
                "profile":         csr_result.get("profile", {}),
                "summary":         csr_result.get("summary", ""),
                "recommendations": csr_result.get("recommendations", []),
                "final_message": (
                    csr_result["messages"][-1].content
                    if csr_result.get("messages") else ""
                ),
                "conversation": agent.get_history()
            })
            break

        response = agent.answer(message)
        user_to_csr.put(response)

print("run_user_sim 정의 완료")

## 5. 단일 세션 실행

In [ ]:
# 페르소나 선택
persona   = PERSONA_TEMPLATES["중년_역사_비문학"]
thread_id = "eval_session_001"

t_csr  = threading.Thread(target=run_csr,      args=(thread_id,))
t_user = threading.Thread(target=run_user_sim,  args=(persona, eval_results))

t_csr.start()
t_user.start()

t_csr.join()
t_user.join()

print("\n" + "="*50)
print("세션 종료")
print("="*50)

## 6. 결과 확인

In [ ]:
if eval_results:
    r = eval_results[-1]

    print("[페르소나]")
    print(json.dumps(r["persona"], ensure_ascii=False, indent=2))

    print("\n[추출된 프로필]")
    print(json.dumps(r["profile"], ensure_ascii=False, indent=2))

    print("\n[요약]")
    print(r["summary"])

    print("\n[책 리스트]")
    print(r["recommendations"])

    print("\n[추천 결과]")
    print(r["final_message"])
else:
    print("결과 없음")

## 7. 다중 페르소나 배치 평가

여러 페르소나를 순차적으로 평가합니다.  
(동시 실행 시 큐 혼선이 생기므로 순차 실행)

In [ ]:
def run_batch_evaluation(persona_dict: dict, base_thread_id: str = "batch"):
    global user_to_csr, csr_to_user
    batch_results = []

    for i, (name, persona) in enumerate(persona_dict.items()):
        print(f"\n{'='*50}")
        print(f"[{i+1}/{len(persona_dict)}] 페르소나: {name}")
        print(f"{'='*50}")

        # 세션마다 큐 초기화
        user_to_csr = queue.Queue()
        csr_to_user = queue.Queue()

        session_results = []
        thread_id = f"{base_thread_id}_{i}"

        t_csr  = threading.Thread(target=run_csr,      args=(thread_id,))
        t_user = threading.Thread(target=run_user_sim,  args=(persona, session_results))

        t_csr.start()
        t_user.start()
        t_csr.join()
        t_user.join()

        if session_results:
            result = session_results[0]
            result["persona_name"] = name
            batch_results.append(result)
            print(f"  완료: 추천 {len(result.get('recommendations', []))}건")

    print(f"\n배치 평가 완료: {len(batch_results)}/{len(persona_dict)} 성공")
    return batch_results


# 실행 (주석 해제 시 전체 페르소나 평가)
# all_results = run_batch_evaluation(PERSONA_TEMPLATES)